# BattINFO Hello World: Your First Linked Battery Record

This notebook is the smallest useful BattINFO workflow for a first-time user.

You will create:

- one cell type
- one physical cell
- one test
- one linked dataset
- one registry-ready submission bundle

The notebook uses a temporary sandbox under `.battinfo/notebooks`, so nothing touches curated repository data.


## 1. Import BattINFO and create a sandbox project


In [ ]:
from pathlib import Path

from battinfo import (
    BatteryCell,
    BatteryCellType,
    Dataset,
    Project,
    Test,
    organization,
    quantity,
)


In [ ]:
project = Project(
    root=Path('.battinfo/notebooks/01-hello-world'),
    name='hello-world',
    title='A123 hello world cycling dataset',
    description='Small sandbox project created by the BattINFO hello world notebook.',
    tenant='digibatt',
    publisher='demo-lab',
    version='0.1.0',
    clean=True,
)

dataset_file = project.root / 'inputs' / 'a123-cycle-life.csv'
dataset_file.parent.mkdir(parents=True, exist_ok=True)
dataset_file.write_text(
    'Cycle Index,Capacity / Ah,Voltage / V\n0,2.50,3.30\n1,2.48,3.28\n2,2.46,3.27\n',
    encoding='utf-8',
)

project.root


## 2. Create the battery objects

This constructor style keeps the first example short. Common fields go directly into the object constructor, and BattINFO still keeps the objects linked underneath.


In [ ]:
cell_design = BatteryCellType(
    manufacturer='A123',
    model='ANR26650M1-B',
    format='cylindrical',
    chemistry='Li-ion',
    size_code='R26650',
    positive_electrode_basis='LFP',
    negative_electrode_basis='unknown',
    nominal_capacity=quantity(2.5, 'Ah'),
    nominal_voltage=quantity(3.3, 'V'),
    diameter=quantity(26.0, 'mm'),
    height=quantity(65.0, 'mm'),
    source={'file': 'a123-anr26650m1-b.manual.json'},
)

cell = BatteryCell(
    cell_design,
    serial_number='hello-world-001',
    batch_id='A123-HELLO-01',
    source={'type': 'lab'},
)

test = Test(
    cell,
    test_type='cycle_life',
    protocol={'name': '1C charge / 1C discharge'},
    instrument='Biologic VSP-300',
    status='completed',
)

dataset = Dataset(
    dataset_file,
    test=test,
    data_format='text/csv',
    license='CC-BY-4.0',
    name='A123 hello world cycling dataset',
    description='Small sandbox dataset created by the BattINFO hello world notebook.',
    creators=[organization('BattINFO Alpha Lab')],
    publisher=organization('BattINFO', url='https://w3id.org/battinfo'),
)

{
    'cell_design': cell_design.model,
    'cell': cell.serial_number,
    'test_type': test.test_type,
    'dataset_path': dataset.path,
    'dataset_format': dataset.data_format,
}


## 3. Add the objects to the project

BattINFO will use the object types to place them into the correct linked record chain.


In [ ]:
project.add(cell_design, cell, test, dataset)


## 4. Publish the linked bundle to RDF/JSON-LD

BattINFO publishes linked battery data as JSON-LD, which is an RDF serialization. For simple file-backed datasets, `project.publish(...)` stages the data into a publication directory for you.


In [ ]:
publish_result = project.publish(dataset)

{
    'jsonld_path': publish_result['publish_path'],
    'triple_count': publish_result['triple_count'],
}


## 5. Build the registry-ready submission bundle

This is the end of the BattINFO-side workflow. `project.bundle()` saves the local project, validates the records, copies the dataset artifact into the project, and writes a registry intake payload for `battinfo-registry`.


In [ ]:
bundle_result = project.bundle()

{
    'registry_intake_path': bundle_result['registry_intake_path'],
    'validation_report_path': bundle_result['validation_report_path'],
    'resource_count': bundle_result['resource_count'],
    'artifact_count': bundle_result['artifact_count'],
}


You have now completed the BattINFO side of the smallest useful workflow.

At this point you have:

- linked BattINFO objects in memory
- a published JSON-LD file
- a saved local project
- a registry-ready intake bundle for `battinfo-registry`

The registry workflow picks up from `dist/registry-intake.json`.
